In [2]:
# Install Groq
!pip install groq -q

from groq import Groq
import json
import time

# ── YOUR API KEY ──────────────────────no────────────────
GROQ_API_KEY = "(your_API_Key)"
client = Groq(api_key=GROQ_API_KEY)

print("✅ Setup complete — Competitor Intelligence Agent ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00
✅ Setup complete — Competitor Intelligence Agent ready


In [3]:
# ── AGENT STEP FUNCTION ───────────────────────────────
def run_agent_step(step_name, system_prompt, user_prompt, previous_output=None):
    """Runs one step of the agent and returns the output."""

    print(f"\n{'='*60}")
    print(f"🤖 AGENT STEP: {step_name}")
    print(f"{'='*60}")
    print("⏳ Processing...")

    start = time.time()

    # If there is previous output, inject it into the user prompt
    if previous_output:
        full_prompt = f"{user_prompt}\n\nContext from previous analysis:\n{previous_output}"
    else:
        full_prompt = user_prompt

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": full_prompt}
        ],
        temperature=0.4,
        max_tokens=1000
    )

    output = response.choices[0].message.content.strip()
    elapsed = time.time() - start

    print(f"✅ Complete in {elapsed:.1f} seconds")
    print(f"\n{output}")

    return output

print("✅ Agent step function ready")

✅ Agent step function ready


In [4]:
# ── STEP 1: MARKET RESEARCHER ─────────────────────────
researcher_system = """You are a senior retail market research analyst
with 10 years of eCommerce experience. You analyse product categories
with precision and back every claim with specific brand examples and
real market knowledge. You are concise, structured, and commercial."""

researcher_task = """Analyse the competitor landscape for the product
category provided.

Identify:
1. The top 5 competitor brands with their price positioning
2. The top 3 positioning strategies being used across the category
3. The dominant features being used as differentiators

Be specific. Use real brand names. Format with clear headings."""

# ── STEP 2: GAP ANALYST ───────────────────────────────
gap_analyst_system = """You are a consumer insights specialist who
identifies underserved needs and market gaps in retail categories.
You think from the customer's perspective and identify frustrations
that existing products do not solve. You are specific and evidence-based."""

gap_analyst_task = """Based on the competitor research above, identify:
1. The 3 biggest underserved customer needs in this category
2. For each gap, explain why existing products fail to address it
3. Rate each gap: High / Medium / Low commercial opportunity

Be specific. Focus on gaps a new entrant could realistically exploit."""

# ── STEP 3: OPPORTUNITY STRATEGIST ───────────────────
strategist_system = """You are a product strategy consultant who
translates market gaps into specific, actionable product opportunities.
You think commercially — every recommendation must be feasible,
differentiated, and tied to a real customer need. You are direct
and specific."""

strategist_task = """Based on the market gaps identified above, generate:
1. Two specific product attribute opportunities a new entrant could exploit
2. For each opportunity:
   - The specific product attributes to focus on
   - The target customer segment
   - The positioning statement in one sentence
   - Why this is commercially viable right now

Be specific enough that a product team could act on this immediately."""

# ── STEP 4: INTELLIGENCE WRITER ───────────────────────
writer_system = """You are a B2B intelligence report writer who
produces competitive briefings for retail and eCommerce clients.
Your reports are clear, structured, and immediately actionable.
You write for senior commercial audiences who need insight,
not just information."""

writer_task = """Using all the research, gap analysis, and strategic
opportunities identified above, write a complete Competitive
Intelligence Brief.

Format it exactly as follows:

COMPETITIVE INTELLIGENCE BRIEF
Category: [category name]
Date: [today's date]

EXECUTIVE SUMMARY
[3 sentences: market context, key finding, top recommendation]

COMPETITOR LANDSCAPE
[Top 5 competitors with positioning]

POSITIONING STRATEGIES
[Top 3 strategies with examples]

MARKET GAPS
[3 gaps with commercial opportunity rating]

STRATEGIC OPPORTUNITIES
[2 opportunities with full detail]

RECOMMENDED ACTION
[One specific, actionable recommendation for immediate execution]

Write in professional B2B tone. Be specific. No filler."""

print("✅ All four agent prompts ready")

✅ All four agent prompts ready


In [5]:
# ── MAIN AGENT FUNCTION ───────────────────────────────
def run_competitor_intelligence_agent(category):
    """
    Runs the full 4-step competitor intelligence agent
    for any product category.
    """

    print("\n" + "="*60)
    print("🚀 COMPETITOR INTELLIGENCE AGENT")
    print(f"📦 Category: {category}")
    print("="*60)
    print("Running 4-step analysis pipeline...\n")

    total_start = time.time()

    # ── STEP 1: MARKET RESEARCH ───────────────────────
    research = run_agent_step(
        step_name="Step 1 — Market Researcher",
        system_prompt=researcher_system,
        user_prompt=f"{researcher_task}\n\nProduct Category: {category}"
    )

    # ── STEP 2: GAP ANALYSIS ──────────────────────────
    gaps = run_agent_step(
        step_name="Step 2 — Gap Analyst",
        system_prompt=gap_analyst_system,
        user_prompt=f"{gap_analyst_task}\n\nProduct Category: {category}",
        previous_output=research
    )

    # ── STEP 3: OPPORTUNITY STRATEGY ──────────────────
    opportunities = run_agent_step(
        step_name="Step 3 — Opportunity Strategist",
        system_prompt=strategist_system,
        user_prompt=f"{strategist_task}\n\nProduct Category: {category}",
        previous_output=f"RESEARCH:\n{research}\n\nGAPS:\n{gaps}"
    )

    # ── STEP 4: WRITE THE BRIEF ───────────────────────
    from datetime import datetime
    today = datetime.now().strftime("%d %B %Y")

    brief = run_agent_step(
        step_name="Step 4 — Intelligence Writer",
        system_prompt=writer_system,
        user_prompt=f"{writer_task}\n\nCategory: {category}\nDate: {today}",
        previous_output=f"""
MARKET RESEARCH:
{research}

GAP ANALYSIS:
{gaps}

STRATEGIC OPPORTUNITIES:
{opportunities}
"""
    )

    total_time = time.time() - total_start

    # ── SUMMARY ───────────────────────────────────────
    print("\n" + "="*60)
    print("✅ AGENT COMPLETE")
    print("="*60)
    print(f"⏱️  Total execution time: {total_time:.1f} seconds")
    print(f"📊 Steps completed: 4/4")
    print(f"📄 Final brief generated: Yes")
    print("="*60)

    return {
        "category": category,
        "research": research,
        "gaps": gaps,
        "opportunities": opportunities,
        "brief": brief,
        "execution_time": total_time
    }

print("✅ Agent runner ready")

✅ Agent runner ready


In [6]:
# ── RUN THE AGENT ─────────────────────────────────────
# Change this category to test any product category

category = "Wireless Earbuds (£30-£80 price range)"

result = run_competitor_intelligence_agent(category)


🚀 COMPETITOR INTELLIGENCE AGENT
📦 Category: Wireless Earbuds (£30-£80 price range)
Running 4-step analysis pipeline...


🤖 AGENT STEP: Step 1 — Market Researcher
⏳ Processing...
✅ Complete in 3.1 seconds

## Competitor Landscape Analysis: Wireless Earbuds (£30-£80)

### Top 5 Competitor Brands with Price Positioning

1. **Anker Soundcore** (£30-£50) - Offers affordable, feature-rich earbuds with a focus on sound quality and battery life.
2. **Samsung** (£50-£70) - Positions itself as a premium brand, offering earbuds with advanced features like wireless charging and seamless integration with Samsung devices.
3. **Beats** (£60-£80) - Targets the high-end segment with stylish, bass-heavy earbuds that appeal to music enthusiasts and fashion-conscious consumers.
4. **Sony** (£50-£70) - Emphasizes its audio expertise, offering earbuds with advanced noise-cancellation and sound quality features.
5. **JBL** (£30-£60) - Focuses on delivering high-quality sound and durable designs at an afford

In [7]:
from google.colab import files
from datetime import datetime
import os

def save_intelligence_brief(result):
    """Save the complete intelligence brief as a text file."""

    os.makedirs("intelligence_briefs", exist_ok=True)

    safe_category = result['category'].replace(" ", "_").replace("/", "-").lower()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"intelligence_briefs/{safe_category}_{timestamp}.txt"

    content = f"""
================================================================
COMPETITIVE INTELLIGENCE BRIEF
Category: {result['category']}
Generated: {datetime.now().strftime("%d %B %Y, %H:%M")}
Execution Time: {result['execution_time']:.1f} seconds
================================================================

--- STEP 1: MARKET RESEARCH ---
{result['research']}

--- STEP 2: GAP ANALYSIS ---
{result['gaps']}

--- STEP 3: STRATEGIC OPPORTUNITIES ---
{result['opportunities']}

--- STEP 4: FINAL INTELLIGENCE BRIEF ---
{result['brief']}

================================================================
Generated by: eCommerce Competitor Intelligence Agent
Part of: eCommerce AI Transformation Playbook
================================================================
"""

    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)

    print(f"✅ Brief saved: {filename}")
    files.download(filename)
    return filename

# Save and download the result
save_intelligence_brief(result)

✅ Brief saved: intelligence_briefs/wireless_earbuds_(£30-£80_price_range)_20260705_051725.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'intelligence_briefs/wireless_earbuds_(£30-£80_price_range)_20260705_051725.txt'

In [8]:
# ── TEST WITH A SECOND CATEGORY ───────────────────────
# This proves the agent works across different product types

category_2 = "Robot Vacuum Cleaners (£150-£300 price range)"

result_2 = run_competitor_intelligence_agent(category_2)
save_intelligence_brief(result_2)


🚀 COMPETITOR INTELLIGENCE AGENT
📦 Category: Robot Vacuum Cleaners (£150-£300 price range)
Running 4-step analysis pipeline...


🤖 AGENT STEP: Step 1 — Market Researcher
⏳ Processing...
✅ Complete in 62.9 seconds

## Competitor Landscape Analysis: Robot Vacuum Cleaners (£150-£300)

### Top 5 Competitor Brands with Price Positioning

1. **Eufy** (£179-£249): Eufy offers a range of robot vacuum cleaners with advanced features like Wi-Fi connectivity and voice control, positioning themselves as a premium yet affordable option.
2. **Shark** (£199-£299): Shark's robot vacuum cleaners are priced at a premium, focusing on high-quality suction power and advanced navigation systems, targeting customers willing to pay more for superior performance.
3. **iRobot (Roomba)** (£249-£299): As a pioneer in the robot vacuum cleaner market, iRobot's Roomba series is positioned at the higher end of the price spectrum, emphasizing advanced features like self-emptying dustbins and smart mapping technology.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'intelligence_briefs/robot_vacuum_cleaners_(£150-£300_price_range)_20260705_051845.txt'